In this tutorial/guide, we will be exploring the different tools to allow Claude to access info from the outside world

When users ask Claude/AI models for current info, it might not respond with the up-to-date or accurate information, or it might just say something like "I'm sorry but I don't have xxx information".

With tools use, you can send Claude/AI models a question along with instructions on how to get extra data from external sources

Claude analyzes the question and decides it needs additional info, then asks for specific details about what data it needs

Your server runs code to fetch the requested info from external APIs or databases

You send the retrieved data back to Claude/AI models which then generates a complete response using both the original question and up-to-date information.

Let's build a project to teach Claud how to set reminders for future dates. Other than the current date, we also need to address the following issues:

- Claude might know the current date but not exact time
- Claude doesn't always handle time-based addition well, especially for future events
- Claude doesn't know how to set a reminder. It doesn't have the permission or built-in mechanism for this



To address the above, we can write a tool function. It is a plain Python function that gets executed automatically when Claude decides it needs extra info to help the user.

The following are guidelines when writing tool functions:

- Use descriptive names
- Validate inputs
- Provide meaningful error messages

The following is an example of a function to get the current date and time.

In [ ]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

We can test with different formats. The validation check ensures Claude cannot pass an empty string for the date format and also teaches Claude if it makes a mistake

In [ ]:
# Default format: "2024-01-15 14:30:25"
get_current_datetime()

# Just hour and minute: "14:30"
get_current_datetime("%H:%M")

JSON Schema tells Claude what arguments the functions expects and how to use it. It essentially acts as documentation for Claude to understand when and how to call your tools.

The complete tool specification has three main parts:

- Name -> clear, descriptive name for your tool (e.g. = "get_weather")
- Description -> what the tool does, when to use it, and what it returns
- Input Schema -> actual JSON schema describing the function's arguments

Instead of writing JSON schemas from scratch, you can use an AI model to generate them. Here's the process:

1. Copy your tool function code
2. Construct a prompt for an AI model to ask it to write a JSON schema for tool calling. Something like "Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation."
3. Include some documentation, e.g.Anthropic documentation on tool use as context
4. Let the AI model generate a properly formatted schema following best practices

Once it generates the schema, copy and paste it to your code. The following shows best practice when it comes to naming functions, e.g.: function_name and function_name_schema

In [ ]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

We can also add

```
ToolParam
```
to catch any type errors in the code before making the api call



In [ ]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    # ... rest of schema
})

To enable Claude to use tools, we need to include a
```
tools
```
parameter in the API call. For example:



In [ ]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

When Claude decides to use a tool, it returns an assistant message with multiple blocks in the content list. It typically contains the following:

- Text Block: Human-readable text explaining what Claude is doing

- ToolUse Block: Instructions for your code about which tool to call and what parameters to use. It includes:
1. An ID for tracking the tool call
2. The name of the function to call (e.g. "get_current_datetime")
3. Input parameters formatted as a dictonary
4. The type designation "tool_use"

When working with tool responses, we must preserve the entire content structure including all blocks. The following is how to append a multi-block assistant message to our conversation history:








In [ ]:
messages.append({
    "role": "assistant",
    "content": response.content
})

After Claude requests a tool call, we need to execute the function and send the results back.

When Claude responds with a tool use block, we extract the input parameters and call the function. Here's how to access the tool parameters

In [ ]:
response.content[1].input
get_current_datetime(**response.content[1].input)


After running the tool function, we need to send the results back to Claude using a tool result block. This block goes inside a user message and tells Claude what happened when we executed the tool.

The tool result block has the following properties:

- tool_use_id = must match the id of the ToolUse block that this ToolResult corresponds to

- content = output from running the tool (serialized as a string)

- is_error = true if an error occurred



and then include the new tool result in the complete conversation history

In [ ]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[1].id,
        "content": "15:04:22",
        "is_error": False
    }]
})

when sending the follow up request, we still need to include the tool schema even though we are not expecting Claude to make another tool call. Claude needs the schema to understand the tool references in the conversation history

In [ ]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

This tool use workflow is now completed. Claude can now access real-time information through our custom function.

We also need to handle scenarios where Claude might need to call several tools in sequence to answer a single user question.

For example:

1. User asks: What day is 103 days from today?

2. Claude responds with a tool use block requesting get_current_datetime

3. Your server calls the function and returns the result

4. Claude realizes it needs more info and requests add_duration_to_datetime

5. Your server calls that function and returns the result

6. Claude now has enough information to provide the final answer

To handle this pattern, we need a conversation loop that continues until Claude stops requesting tools. Something like the following:

In [ ]:
def run_conversation(messages):
    while True:
        response = chat(messages)

        add_assistant_message(messages, response)

        # Pseudo code
        if response isn't asking for a tool:
            break

        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)

    return messages

Before implementing the conversation loop, you need to update the helper functions to handle multiple message blocks properly. For example we can update the functions to handle full message objects:

In [ ]:
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

and then modifying the chat function to accept the list of tools and return the full message instead of just text

In [ ]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

Since we are now retuning full message objects, we to create a helper function to extract text when needed

In [ ]:
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

Also, another way to know if Claude wants to use a tool is in the stop_Reason field of the response message. When Claude decides it needs to call a tool, the stop_reason field gets set to "tool_use".
So basically, we can add a check in our loop.

In [ ]:
response.stop_reason != "tool_use":
    break  # Claude is done, no more tools needed

In [ ]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

Claude can request multiple tools in a single response. The message content contains a list of blocks, and we need to process each tool use block separately.

The run_tools function handles this by filtering for tool use blocks and processing each one:

In [ ]:
def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []

    for tool_request in tool_requests:
        # Process each tool request...

Each tool use block must be answered with a corresponding tool result block. The connection between them is maintained though matching IDs.

In [ ]:
tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": json.dumps(tool_output),
    "is_error": False
}

We also need to account for errors when using tools. If a tool fairs, we still need to provide a result block to Claude.

In [ ]:
try:
    tool_output = run_tool(tool_request.name, tool_request.input)
    tool_result_block = {
        "type": "tool_result",
        "tool_use_id": tool_request.id,
        "content": json.dumps(tool_output),
        "is_error": False
    }
except Exception as e:
    tool_result_block = {
        "type": "tool_result",
        "tool_use_id": tool_request.id,
        "content": f"Error: {e}",
        "is_error": True
    }

To support multiple tools, create a routing function that maps tool names to their implementations

In [ ]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "another_tool":
        return another_tool(**tool_input)
    # Add more tools as needed